# Experiment 13: Recurrent Neural Networks (RNN) and Long Short-Term Memory (LSTM)

**Objective**: To implement, train, and compare Recurrent Neural Networks (RNN) and Long Short-Term Memory (LSTM) networks using the Keras library on the IMDB movie review sentiment classification dataset.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import warnings

warnings.filterwarnings('ignore')

## 1. Data Loading and Preprocessing

We use the IMDB dataset, which contains 50,000 highly polarized movie reviews for binary sentiment classification (positive or negative). We will limit our vocabulary to the top 10,000 most frequent words.
Then, we pad the sequences so that all input vectors have the same length (500 words).

In [ ]:
vocab_size = 10000
maxlen = 500  # maximum length of a review

# Load data
print("Loading data...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
print(f"{len(x_train)} train sequences")
print(f"{len(x_test)} test sequences")

# Pad sequences
print("Pad sequences (samples x time)...")
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

## 2. Simple RNN Model

First, we build a model using a standard `SimpleRNN` layer. The SimpleRNN layer struggles with long-term dependencies due to the vanishing gradient problem, but it serves as a good baseline.

In [ ]:
rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=maxlen),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.summary()

### Training the Simple RNN

In [ ]:
epochs = 5
batch_size = 128

print("Training Simple RNN...")
rnn_history = rnn_model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

In [ ]:
print("Evaluating Simple RNN on test data...")
rnn_loss, rnn_acc = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Simple RNN Test Accuracy: {rnn_acc:.4f}")

## 3. LSTM Model

Next, we build a model using an `LSTM` layer. Long Short-Term Memory networks are designed specifically to overcome the vanishing gradient problem, allowing them to learn long-range dependencies effectively.

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=maxlen),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

### Training the LSTM

In [ ]:
print("Training LSTM...")
lstm_history = lstm_model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

In [ ]:
print("Evaluating LSTM on test data...")
lstm_loss, lstm_acc = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"LSTM Test Accuracy: {lstm_acc:.4f}")

## 4. Comparison and Visualization

We plot the training and validation accuracy of both models side-by-side.

In [ ]:
plt.figure(figsize=(14, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(rnn_history.history['val_accuracy'], label='RNN Validation Accuracy', color='blue')
plt.plot(lstm_history.history['val_accuracy'], label='LSTM Validation Accuracy', color='green')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(rnn_history.history['val_loss'], label='RNN Validation Loss', color='blue', linestyle='--')
plt.plot(lstm_history.history['val_loss'], label='LSTM Validation Loss', color='green', linestyle='--')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

## 5. Conclusion

- **Simple RNNs** tend to suffer from vanishing gradients when sequence lengths are long (like 500 words). As a result, they may plateau in accuracy very quickly or start overfitting.
  
- **LSTMs** handle longer sequences significantly better due to their gated cell state, which allows them to decide what information to keep, what to throw away, and what to update, leading to superior final performance on such sentiment analysis tasks.